# 02 · Data Exploration

The eight queries use a single snapshot produced by **01 · Data Preparation**. Each complete result is exported as Parquet. Previews are bounded and the summary identifies the snapshot and the parameters used.

In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = next((path for path in (cwd, *cwd.parents) if (path / "src/imdb_loader.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run the notebook from the IMDb repository or its notebooks folder.")
sys.path.insert(0, str(ROOT))

import hashlib
import json
from uuid import uuid4

from pyspark import StorageLevel
from pyspark.sql import functions as F
from src import exploration as query
from src.imdb_loader import data_root, prepared_snapshot, read_prepared, spark_session

DATA = data_root(ROOT)
snapshot = prepared_snapshot(DATA)
validation_year = json.loads((snapshot / "manifest.json").read_text(encoding="utf-8"))["validation_year"]
spark = spark_session("IMDb Data Exploration")
tables = read_prepared(spark, snapshot)
titles, ratings, people = tables["title.basics"], tables["title.ratings"], tables["name.basics"]
crew, principals = tables["title.crew"], tables["title.principals"]
films = query.movies(titles, ratings).persist(StorageLevel.DISK_ONLY)

PARAMETERS = {
    "q01": dict(min_minutes=40, min_genre_films=30, min_robust_z=3.5),
    "q02": dict(first_year=1980, last_year=2025),
    "q03": dict(top_n=3),
    "q04": dict(high_rating=8.0, max_other_mean=6.0, min_other_films=2),
    "q05": dict(min_seasons=3, min_episodes=10, min_votes=1),
    "q06": dict(min_rating=7.0, min_votes=1000),
    "q07": dict(through_year=validation_year, min_years=20, min_films=15, early_years=5, min_votes=100),
    "q08": dict(min_votes=100),
}

results_dir = DATA / "results" / snapshot.name / uuid4().hex
results_dir.mkdir(parents=True)
counts = {}


def save_result(name, frame, order, invalid, *, preview=12, display_columns=None):
    """Materialize each expensive query once, then verify and display its export."""
    path = (results_dir / name).as_posix()
    frame.write.parquet(path)
    result = spark.read.parquet(path)
    assert not result.filter(F.coalesce(invalid, F.lit(True))).take(1), f"Invalid invariant: {name}"
    counts[name] = result.count()
    print(f"{name} : {counts[name]:,} lignes")
    sample = result.orderBy(*order)
    if display_columns is not None:
        sample = sample.select(*display_columns)
    sample.show(preview, truncate=55)
    return result

## Q1 · Atypical runtimes by genre

The query uses `title.basics` and the shared film view. A feature film is defined here as an IMDb title of type `movie` lasting at least 40 minutes. Deviations in both directions are kept for genres with at least 30 films. A film is selected when its modified z-score, 0.6745 × (runtime − genre median) / genre MAD, is at least 3.5 in absolute value (Iglewicz and Hoaglin). The genre mean, standard deviation and classic z-score are exported alongside for comparison: a few runtimes lasting several days inflate the standard deviation of whole genres, which the median and MAD are not sensitive to. A title can appear in several genres.

In [ ]:
q1 = PARAMETERS["q01"]
runtime = query.runtime_outliers(films, **q1)
runtime = save_result("q01_runtime_outliers", runtime,
                      [F.abs("robust_z").desc(), "genre", "tconst"],
                      (F.col("runtimeMinutes") < q1["min_minutes"])
                      | (F.col("genre_films") < q1["min_genre_films"])
                      | (F.col("genre_mad_minutes") <= 0)
                      | (F.abs("robust_z") < q1["min_robust_z"]),
                      display_columns=["tconst", "primaryTitle", "startYear", "genre", "runtimeMinutes",
                                       "genre_median_minutes", "genre_mad_minutes", "robust_z",
                                       "genre_mean_minutes", "z_score"])
runtime.groupBy("genre").agg(
    F.count("*").alias("outliers"),
    F.count_if(F.col("robust_z") > 0).alias("longer"),
    F.count_if(F.col("robust_z") < 0).alias("shorter"),
    F.first("genre_median_minutes").alias("median_minutes"),
    F.first("genre_mad_minutes").alias("mad_minutes"),
).orderBy(F.desc("outliers")).show(30, truncate=False)

## Q2 · Three most popular genres per year

The query uses `title.basics`, `title.ratings`. Popularity is the sum of current IMDb votes for films released each year from 1980 to 2025 inclusive. A film contributes to each of its genres. Films without a valid rating and vote count are included in the film volume. Ties are broken by genre name.

In [ ]:
q2, q3 = PARAMETERS["q02"], PARAMETERS["q03"]
ranked = query.genre_rankings(films, **q2).persist(StorageLevel.MEMORY_AND_DISK)
top = save_result("q02_top_genres", ranked.filter(F.col("rank") <= q3["top_n"]),
                  ["year", "rank"], (F.col("rank") < 1) | (F.col("rank") > q3["top_n"])
                  | ~F.col("year").between(q2["first_year"], q2["last_year"])
                  | (F.col("rated_films") > F.col("films")) | (F.col("votes") < 0),
                  display_columns=["year", "rank", "genre", "votes", "films", "rated_films"])
year_sizes = top.groupBy("year").count()
assert not year_sizes.filter(F.col("count") > q3["top_n"]).take(1)
expected_years = spark.range(q2["first_year"], q2["last_year"] + 1).select(F.col("id").alias("year"))
gaps = expected_years.join(year_sizes, "year", "left").filter(
    F.coalesce(F.col("count"), F.lit(0)) < q3["top_n"]
).count()
print(f"Years with fewer than {q3['top_n']} genres: {gaps}")

## Q3 · Dominant genres versus other genres

The query uses `title.basics`, `title.ratings`. For each year, we compare the vote share, the votes per film–genre pair, the number of pairs and the vote-weighted rating. The per-genre detail makes it possible to recover the ranking and the underlying volumes.

In [ ]:
detail = save_result("q03_genre_detail", ranked, ["year", "rank"],
                     (F.col("films") < F.col("rated_films")) | (F.col("rank") < 1))
comparison = query.compare_genres(ranked, **q3)
comparison = save_result("q03_top_vs_rest", comparison, ["year", F.desc("group")],
                         (F.col("genres") < 1) | (F.col("rated_film_genre_pairs") > F.col("film_genre_pairs"))
                         | ~F.coalesce(F.col("vote_share").between(0.0, 1.0), F.lit(True)))
assert not comparison.groupBy("year").agg(F.sum("vote_share").alias("share")).filter(
    F.abs(F.col("share") - 1.0) > 1e-9
).take(1)
ranked.unpersist()

## Q4 · Director performance

The query uses `title.crew`, `title.basics`, `title.ratings`, `name.basics`. Each director–film pair counts once, including co-directed films. A director needs exactly one film rated at least 8 and at least two other rated films whose arithmetic mean is at most 6. Unrated films are counted separately.

In [ ]:
q4 = PARAMETERS["q04"]
directors = save_result("q04_directors", query.directors_performance(films, crew, people, **q4),
                        [F.desc("other_rated_films"), "nconst"],
                        (F.col("high_films") != 1) | (F.col("high_film.averageRating") < q4["high_rating"])
                        | (F.col("other_rated_films") < q4["min_other_films"])
                        | (F.col("other_mean") > q4["max_other_mean"]))

## Q5 · Series evolution

The query uses `title.episode`, `title.basics`, `title.ratings`. Series and miniseries must have at least three numbered seasons, each containing ten or more episodes. The last season is the highest observed number. The extremes use the available ratings across all numbered seasons. In case of a tie, every episode at the minimum must be in the last season and every episode at the maximum in season 1 or 2. The table also shows rating coverage.

In [ ]:
q5 = PARAMETERS["q05"]
series = save_result("q05_series", query.series_evolution(titles, tables["title.episode"], ratings, **q5),
                     [F.desc("qualifying_seasons"), "parentTconst"],
                     (F.col("qualifying_seasons") < q5["min_seasons"])
                     | (F.col("worst_episode.seasonNumber") != F.col("last_season"))
                     | (F.col("best_episode.seasonNumber") > 2)
                     | (F.col("rated_episodes") > F.col("episodes")))

## Q6 · Success in several languages

The query uses `title.basics`, `title.ratings`, `title.principals`, `name.basics` and an optional reference of original languages. Success is defined as an IMDb rating of at least 7 and at least 1,000 votes. Principal actor and actress credits are deduplicated per film and person.

The public files provide no spoken language. With `data/reference/original_languages.csv`, only documented original languages are used and one provenance URL is kept per example. Otherwise, a separate indicator is computed from the explicit and consistent languages of titles flagged as original in `title.akas`. This indicator describes the language of the title and its coverage is measured. It cannot establish that a film or an actor is multilingual. At least two distinct films are required. The section separately exports people associated with at least two and at least three languages.

[IMDb public schema](https://data.imdb.com/non-commercial-datasets/) · [Wikidata – original language](https://www.wikidata.org/wiki/Property:P364) · [Wikidata – IMDb ID](https://www.wikidata.org/wiki/Property:P345)

In [ ]:
actors = query.acting_credits(principals).join(films.select("tconst"), "tconst", "left_semi").persist(
    StorageLevel.DISK_ONLY
)
q6 = PARAMETERS["q06"]
successful = films.filter(
    (F.col("averageRating") >= q6["min_rating"]) & (F.col("numVotes") >= q6["min_votes"])
)
reference_path = DATA / "reference" / "original_languages.csv"
reference_hash = None
if reference_path.is_file():
    language_mode = "verified"
    with reference_path.open("rb") as stream:
        reference_hash = hashlib.file_digest(stream, "sha256").hexdigest()
    language_rows = query.verified_film_languages(spark, reference_path)
else:
    language_mode = "title_proxy"
    language_rows = query.original_title_languages(tables["title.akas"])
    print("Spoken languages unavailable. The export covers the languages of original titles.")
    print(f"Run 03_data_enrichment.ipynb first to create {reference_path} "
          "(Wikidata original languages) and switch Q6 to the verified mode.")

languages = language_rows.join(successful.select("tconst"), "tconst", "left_semi").persist(StorageLevel.DISK_ONLY)
coverage = successful.select("tconst").join(
    languages.select("tconst").distinct().withColumn("covered", F.lit(True)), "tconst", "left"
).agg(F.count("*").alias("successful_films"), F.count("covered").alias("films_with_language")).withColumn(
    "mode", F.lit(language_mode)
).withColumn("coverage", F.try_divide("films_with_language", "successful_films"))
coverage = save_result("q06_language_coverage", coverage, ["mode"],
                       F.col("films_with_language") > F.col("successful_films"))

multi = query.multilingual_actors(successful, actors, people, languages)
multi = save_result(f"q06_{language_mode}_at_least_2", multi, [F.desc("language_count"), "nconst"],
                    (F.col("language_count") < 2) | (F.col("language_count") != F.size("languages")))
multi3 = save_result(f"q06_{language_mode}_at_least_3", multi.filter(F.col("language_count") >= 3),
                     [F.desc("language_count"), "nconst"], F.col("language_count") < 3)
languages.unpersist()

## Q7 · Early-career popularity peak

The query uses `title.basics`, `title.ratings`, `title.principals`, `name.basics`. The observable career is the span between the first and last dated films among the principal credits. It must cover at least 20 years and 15 films. The early period covers the first five calendar years. The maximum rating of those films with at least 100 votes must be strictly greater than the later maximum. An eligible rating is required in each period.

In [ ]:
q7 = PARAMETERS["q07"]
peaks = save_result("q07_early_peaks", query.early_career_peaks(films, actors, people, **q7),
                    [F.desc("early_peak.averageRating"), F.desc("early_peak.numVotes"), "nconst"],
                    (F.col("career_years") < q7["min_years"]) | (F.col("dated_films") < q7["min_films"])
                    | (F.col("early_peak.startYear") >= F.col("first_year") + q7["early_years"])
                    | (F.col("early_peak.averageRating") <= F.col("later_peak_rating")))
actors.unpersist()

## Q8 · Professionals with multiple roles

The query uses `title.basics`, `title.ratings`, `title.crew`, `title.principals`, `name.basics`. Principal credits are combined with the directing and writing lists. The actor and actress categories represent the same profession. Self appearances and archive footage are excluded. At least two professions are required over the whole observable career. For the same person, we compare the average ratings of their single-role films and multi-role films. The averages cover films with at least 100 votes. The difference stays empty when an average is unavailable.

The summary table aggregates these per-person differences: one row for every multi-role film (`any`) and one row per combination of core roles held on the same film (actor, director, writer). It reports the number of people, the mean and median rating difference and the share of people rated higher, or lower, when they hold several roles.

In [ ]:
q8 = PARAMETERS["q08"]
professionals = save_result("q08_professionals",
                            query.multi_role_performance(films, crew, principals, people, **q8),
                            [F.desc("multi_rated_films"), "nconst"],
                            (F.size("career_roles") < 2)
                            | (F.col("multi_rated_films") > F.col("multi_role_films"))
                            | (F.col("single_rated_films") > F.col("single_role_films")),
                            display_columns=["nconst", "primaryName", "career_roles", "multi_role_films",
                                             "single_role_films", "multi_mean_rating", "single_mean_rating",
                                             "rating_delta"])
q8_summary = save_result("q08_summary",
                         query.multi_role_summary(films, crew, principals, **q8),
                         [F.desc("people")],
                         (F.col("people") < 1)
                         | ~F.col("share_better_multi").between(0.0, 1.0)
                         | ~F.col("share_worse_multi").between(0.0, 1.0),
                         preview=20)
films.unpersist()

## Export index

Each subfolder contains the complete DataFrame. The `summary.json` file lists the volumes and thresholds once every export and check has succeeded.

In [ ]:
summary = {
    "prepared_snapshot": snapshot.name,
    "parameters": PARAMETERS,
    "q06_language_mode": language_mode,
    "q06_reference_sha256": reference_hash,
    "q06_coverage": coverage.first().asDict(),
    "q02_years_with_fewer_than_top_n": gaps,
    "rows": counts,
}
(results_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(results_dir)
print(json.dumps(counts, indent=2))
spark.stop()